# DAIC-WOZ Participant Audio Isolation

This notebook creates participant-only WAV files from each DAIC-WOZ session.

**Goal**
- Read each session's `*_AUDIO.wav` and `*_TRANSCRIPT.csv`.
- Keep only transcript rows where `speaker == Participant`.
- Skip `scrubbed_entry` rows.
- Stitch the participant segments together in timestamp order.
- Save the result as `processed/isolated_audio/<session_name>.wav`.

Example: `300_P.zip` -> `processed/isolated_audio/300_P.wav`


## 1. Imports And Configuration

In [1]:
from __future__ import annotations

import csv
import io
import re
import shutil
import tempfile
import wave
import zipfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

BASE_DIR = Path('.')
DATASET_DIR = BASE_DIR / 'dataset'
PROCESSED_DIR = BASE_DIR / 'processed'
OUTPUT_DIR = PROCESSED_DIR / 'isolated_audio'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPEAKER_TO_KEEP = 'participant'
TRANSCRIPT_SUFFIX = '_TRANSCRIPT.csv'
AUDIO_SUFFIX = '_AUDIO.wav'

INTERRUPTION_RANGES = {
    373: (5 * 60 + 52, 7 * 60 + 0),
    444: (4 * 60 + 46, 6 * 60 + 27),
}

print(f'Dataset dir : {DATASET_DIR.resolve()}')
print(f'Output dir  : {OUTPUT_DIR.resolve()}')
print(f'Speaker kept: {SPEAKER_TO_KEEP}')

Dataset dir : /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/dataset
Output dir  : /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/processed/isolated_audio
Speaker kept: participant


## 2. Discover Sessions

This notebook expects session ZIP files like `dataset/300_P.zip`.


In [2]:
@dataclass
class SessionSource:
    session_name: str
    participant_id: int
    path: Path

def parse_session_name(name: str) -> tuple[str, int] | None:
    match = re.fullmatch(r'(\d+_P)', name)
    if match is None:
        return None
    session_name = match.group(1)
    participant_id = int(session_name.split('_')[0])
    return session_name, participant_id

session_list = []

for path in sorted(DATASET_DIR.glob('*.zip')):
    parsed = parse_session_name(path.stem)
    if parsed is None:
        continue
    session_name, participant_id = parsed
    session_list.append(SessionSource(session_name, participant_id, path))

print(f'Found {len(session_list)} session sources')
print('First 10 sessions:', [s.session_name for s in session_list[:10]])

Found 189 session sources
First 10 sessions: ['300_P', '301_P', '302_P', '303_P', '304_P', '305_P', '306_P', '307_P', '308_P', '309_P']


## 3. Helper Functions

In [3]:
def extract_required_files_from_zip(zip_path: Path, temp_dir: Path) -> tuple[Path, Path]:
    wav_path = None
    transcript_path = None
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for member in zf.namelist():
            member_name = Path(member).name
            if member_name.endswith(AUDIO_SUFFIX):
                zf.extract(member, temp_dir)
                wav_path = temp_dir / member
            elif member_name.endswith(TRANSCRIPT_SUFFIX):
                zf.extract(member, temp_dir)
                transcript_path = temp_dir / member
    if wav_path is None or transcript_path is None:
        raise FileNotFoundError(f'Missing required files inside {zip_path.name}')
    return wav_path, transcript_path

def load_transcript_rows(transcript_path: Path) -> list[dict]:
    try:
        with transcript_path.open('r', encoding='utf-8', newline='') as f:
            reader = csv.DictReader(f, delimiter='\t')
            rows = list(reader)
        if rows and {'start_time', 'stop_time', 'speaker', 'value'}.issubset(rows[0].keys()):
            return rows
    except Exception:
        pass

    rows = []
    with transcript_path.open('r', encoding='utf-8') as f:
        text = f.read()
    reader = csv.DictReader(io.StringIO(text), delimiter='\t')
    for row in reader:
        rows.append(row)
    return rows

def participant_turns_from_transcript(rows: list[dict], participant_id: int) -> list[tuple[float, float]]:
    turns = []
    for row in rows:
        speaker = str(row.get('speaker', '')).strip().lower()
        value = str(row.get('value', '')).strip().lower()
        if speaker != SPEAKER_TO_KEEP:
            continue
        if value == 'scrubbed_entry':
            continue

        start_time = float(row['start_time'])
        stop_time = float(row['stop_time'])
        if stop_time <= start_time:
            continue
        turns.append((start_time, stop_time))

    if participant_id in INTERRUPTION_RANGES:
        int_start, int_end = INTERRUPTION_RANGES[participant_id]
        filtered = []
        for start_time, stop_time in turns:
            overlaps = start_time < int_end and stop_time > int_start
            if not overlaps:
                filtered.append((start_time, stop_time))
        turns = filtered

    return turns

def read_wav_pcm16_mono(wav_path: Path) -> tuple[np.ndarray, dict]:
    with wave.open(str(wav_path), 'rb') as wf:
        params = {
            'nchannels': wf.getnchannels(),
            'sampwidth': wf.getsampwidth(),
            'framerate': wf.getframerate(),
            'nframes': wf.getnframes(),
            'comptype': wf.getcomptype(),
            'compname': wf.getcompname(),
        }
        audio_bytes = wf.readframes(wf.getnframes())

    if params['sampwidth'] != 2:
        raise ValueError(f"Expected 16-bit PCM WAV, got sample width {params['sampwidth']}")

    audio = np.frombuffer(audio_bytes, dtype=np.int16)
    if params['nchannels'] > 1:
        audio = audio.reshape(-1, params['nchannels'])[:, 0]
    return audio.copy(), params

def stitch_turns(audio: np.ndarray, sample_rate: int, turns: list[tuple[float, float]]) -> np.ndarray:
    chunks = []
    for start_time, stop_time in turns:
        start_sample = max(0, int(round(start_time * sample_rate)))
        stop_sample = min(len(audio), int(round(stop_time * sample_rate)))
        if stop_sample <= start_sample:
            continue
        chunks.append(audio[start_sample:stop_sample])
    if not chunks:
        return np.zeros(0, dtype=np.int16)
    return np.concatenate(chunks).astype(np.int16, copy=False)

def write_wav_pcm16(output_path: Path, audio: np.ndarray, params: dict):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(output_path), 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(params['framerate'])
        wf.writeframes(audio.astype(np.int16, copy=False).tobytes())

## 4. Single-Session Sanity Check

This cell processes one session so you can confirm the output format before running all sessions.


In [4]:
def process_session(session: SessionSource, output_dir: Path) -> dict:
    temp_root = None
    try:
        temp_root = Path(tempfile.mkdtemp(prefix=f'{session.session_name}_'))
        wav_path, transcript_path = extract_required_files_from_zip(session.path, temp_root)

        transcript_rows = load_transcript_rows(transcript_path)
        turns = participant_turns_from_transcript(transcript_rows, session.participant_id)
        audio, params = read_wav_pcm16_mono(wav_path)
        stitched_audio = stitch_turns(audio, params['framerate'], turns)

        output_path = output_dir / f'{session.session_name}.wav'
        write_wav_pcm16(output_path, stitched_audio, params)

        return {
            'session_name': session.session_name,
            'source_path': str(session.path),
            'output_path': str(output_path),
            'sample_rate': params['framerate'],
            'original_duration_s': round(len(audio) / params['framerate'], 3),
            'isolated_duration_s': round(len(stitched_audio) / params['framerate'], 3),
            'num_turns': len(turns),
        }
    finally:
        if temp_root is not None:
            shutil.rmtree(temp_root, ignore_errors=True)

sample_session = next(s for s in session_list if s.session_name == '300_P')
sample_result = process_session(sample_session, OUTPUT_DIR)
sample_result

{'session_name': '300_P',
 'source_path': 'dataset/300_P.zip',
 'output_path': 'processed/isolated_audio/300_P.wav',
 'sample_rate': 16000,
 'original_duration_s': 648.5,
 'isolated_duration_s': 155.76,
 'num_turns': 87}

## 5. Verify The Saved Sample

In [5]:
sample_output = OUTPUT_DIR / '300_P.wav'
with wave.open(str(sample_output), 'rb') as wf:
    print('output:', sample_output)
    print('channels:', wf.getnchannels())
    print('sample_width:', wf.getsampwidth())
    print('sample_rate:', wf.getframerate())
    print('duration_sec:', wf.getnframes() / wf.getframerate())

output: processed/isolated_audio/300_P.wav
channels: 1
sample_width: 2
sample_rate: 16000
duration_sec: 155.76


## 6. Process All Sessions

This will create one participant-only WAV per session under `processed/isolated_audio/`.


In [6]:
results = []
errors = []

for session in tqdm(session_list, desc='Isolating participant audio'):
    try:
        results.append(process_session(session, OUTPUT_DIR))
    except Exception as exc:
        errors.append({'session_name': session.session_name, 'error': str(exc)})

results_df = np.nan
if results:
    import pandas as pd
    results_df = pd.DataFrame(results).sort_values('session_name').reset_index(drop=True)
    display(results_df.head())
    print(f'Processed sessions: {len(results_df)}')
    print(f'Total isolated hours: {results_df["isolated_duration_s"].sum() / 3600:.2f}')

if errors:
    import pandas as pd
    error_df = pd.DataFrame(errors)
    display(error_df)
    print(f'Errors: {len(error_df)}')
else:
    print('No processing errors encountered.')

Isolating participant audio:   0%|          | 0/189 [00:00<?, ?it/s]

,session_name,source_path,output_path,sample_rate,original_duration_s,isolated_duration_s,num_turns
0,300_P,dataset/300_P.zip,processed/isolated_audio/300_P.wav,16000,648.5,155.760,87
1,301_P,dataset/301_P.zip,processed/isolated_audio/301_P.wav,16000,823.9,475.440,104
2,302_P,dataset/302_P.zip,processed/isolated_audio/302_P.wav,16000,758.8,208.933,97
3,303_P,dataset/303_P.zip,processed/isolated_audio/303_P.wav,16000,985.3,642.930,103
4,304_P,dataset/304_P.zip,processed/isolated_audio/304_P.wav,16000,792.6,362.600,104


Processed sessions: 189
Total isolated hours: 24.30
No processing errors encountered.


## 7. Save A Processing Summary

In [7]:
summary_path = OUTPUT_DIR / 'isolation_summary.csv'
if isinstance(results_df, np.ndarray):
    print('No results dataframe to save yet.')
else:
    results_df.to_csv(summary_path, index=False)
    print(f'Saved summary to: {summary_path.resolve()}')

Saved summary to: /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/processed/isolated_audio/isolation_summary.csv
